# 1D ResNet U-Net sleep detection

Run the shared package pipeline: prepare day windows, train by subject, evaluate
held-out predictions, then predict test events. Put the competition files at the
repository root. Choose a fresh checkpoint directory for each training run.
OOF scoring uses plain peaks; learned time priors are used only for test prediction.


In [ ]:
import sys
from pathlib import Path

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from sleep_unet.cli import main
from sleep_unet.data import prepare_windows, read_table, save_prepared
from sleep_unet.metrics import score_events
from sleep_unet.postprocess import scores_to_long, submission_without_priors
from sleep_unet.train import load_oof, rmse, train_model


In [ ]:
SERIES_PATH = ROOT / "train_series.parquet"
EVENTS_PATH = ROOT / "train_events.csv"
TEST_PATH = ROOT / "test_series.parquet"
PREPARED_DIR = ROOT / "data/prepared"
CHECKPOINT_DIR = ROOT / "artifacts/notebook"

n_splits = 5
n_epochs = 20
batch_size = 16
features = ["log_anglez_std", "log_enmo", "valid_flag"]
filters = 24
seed = 1  # Number of independent training seeds to average.


## Prepare and train

Windows contain a day plus three hours on either side. Context comes only from
adjacent calendar days. The mask excludes missing and repeated sensor minutes
from both training loss and RMSE. Folds hold out whole series.


In [ ]:
events = read_table(EVENTS_PATH)
prepared = prepare_windows(read_table(SERIES_PATH), events, features=features)
save_prepared(prepared, PREPARED_DIR)
print("X", prepared.X.shape, "Y", prepared.Y.shape)


In [ ]:
result = train_model(
    prepared, CHECKPOINT_DIR, epochs=n_epochs, n_splits=n_splits,
    batch_size=batch_size, filters=filters, seed=seed,
)
print("best fold RMSE", result.best_rmse)


## Evaluate held-out predictions

The OOF cache must match the prepared windows and targets. Event labels enter
only the scorer, so validation labels cannot influence peak selection.


In [ ]:
oof = load_oof(prepared, CHECKPOINT_DIR)
print("OOF RMSE", rmse(prepared.Y, oof, prepared.mask))
submission = submission_without_priors(scores_to_long(prepared, oof))
submission.to_csv(CHECKPOINT_DIR / "oof_submission.csv", index=False)
ap, ap_table, _ = score_events(events, submission)
print("event AP", ap)
print(ap_table.to_string())


## Predict test events

Inference averages the declared seed/fold checkpoints and uses their saved
normalization. The time priors below are learned from training events.


In [ ]:
main([
    "predict", "--series", str(TEST_PATH), "--events", str(EVENTS_PATH),
    "--checkpoints", str(CHECKPOINT_DIR), "--batch-size", str(batch_size),
    "--out", str(CHECKPOINT_DIR / "submission.csv"),
])
